# Image Manipulation and Analysis With `scikit-image`

In this lab, you'll use some of scikit-image's features to manipulate and analyze a photo: correct its contrast, detect its edges, detect and match keypoints, stitch two overlapping images back together, then compare images with metrics. On another image, of coins, you'll also count and measure objects.

1. An image's histogram
2. Correcting the contrast
3. Detecting edges, before and after a blur
4. Thresholding, counting and measuring objects
5. Detecting keypoints
6. Matching keypoints
7. Stitching two overlapping images
8. Image comparison metrics

scikit-image works on NumPy arrays: you need to be at ease with manipulating them (indexing, slices, operations on whole arrays) to make the most of it.

The exercises build on each other: each one uses what the previous ones' solutions define. Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed.

Start by running the next cell, without reading it: it downloads the lab's photo and defines the `side_by_side` function, which displays images side by side.

In [ ]:
# @title Run me: environment setup
# The lab's photo: Ezarate, CC BY-SA 4.0, Wikimedia Commons.
!curl -fsSL -o la-havana.jpg https://upload.wikimedia.org/wikipedia/commons/thumb/7/74/AvMalecon-LaHabanaCuba-04735.jpg/1920px-AvMalecon-LaHabanaCuba-04735.jpg

from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
from skimage import (
  color,
  data,
  exposure,
  feature,
  filters,
  io,
  measure,
  metrics,
  transform,
  util,
)


def side_by_side(
  images: Sequence[np.ndarray],
  title: str | None = None,
  subtitles: Sequence[str] | None = None,
  dpi: int = 150,
) -> None:
  """Display images side by side, each with its subtitle."""
  fig, axes = plt.subplots(1, len(images), dpi=dpi)
  for i, (ax, image) in enumerate(zip(axes, images)):
    # A single-channel image is displayed in grayscale.
    ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
    ax.axis("off")
    if subtitles:
      ax.set_title(subtitles[i], fontsize=8)
  if title:
    fig.suptitle(title, y=0.85)
  fig.tight_layout()
  plt.show()

## 1. An Image's Histogram

The lab's photo, `la-havana.jpg`, which the setup cell downloaded, shows Malecón Avenue in Havana (photo by Ezarate, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/), [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:AvMalecon-LaHabanaCuba-04735.jpg)).

An image's histogram counts its pixels by intensity, from 0 to 255 for a `uint8` image: it shows at a glance whether the image is dark, bright or low-contrast, and the contrast corrections of the next exercise rely on it.

- Load the photo into a variable `im` with `io.imread`, as in the course, and print its shape and type.
- Write the function `show_histogram(image, cumulative=False)`, which computes the histogram of each of the three channels of `image` with `exposure.histogram` and plots the three curves on the same chart, each in its channel's color. With `cumulative=True`, it plots each channel's cumulative histogram: for each intensity, the number of pixels of lower or equal intensity.
- Show the histogram of `im`, then its cumulative histogram. The next exercises use `show_histogram` to compare the histograms before and after a correction.

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.histogram
- https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html

In [ ]:
def show_histogram(image: np.ndarray, cumulative: bool = False) -> None:
  # Your code here
  ...


# Your code here

### Additional hints

- `io.imread("la-havana.jpg")` returns a `uint8` NumPy array of shape `(height, width, channels)`: you should get `(1247, 1920, 3)`.
- `exposure.histogram(image, channel_axis=2)` computes one histogram per channel, the channels being on axis 2. It returns a pair `(histograms, values)`: `histograms` has one row per channel, `(3, 256)` for `im`, and `values` gives the intensities, on the x-axis.
- `np.cumsum(histogram)` gives the cumulative histogram. Its last value is the image's number of pixels, 1247 × 1920 = 2,394,240.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def show_histogram(image: np.ndarray, cumulative: bool = False) -> None:
  histograms, values = exposure.histogram(image, channel_axis=...)
  for histogram, color in zip(histograms, ("red", "green", "blue")):
    if cumulative:
      histogram = ...
    plt.plot(values, histogram, color=color)
  plt.xlabel("Intensity")
  plt.ylabel("Number of pixels")
  plt.show()


im = io.imread(...)
print(im.shape, im.dtype)
show_histogram(im)
show_histogram(im, cumulative=True)
```

### Solution

In [ ]:
def show_histogram(image: np.ndarray, cumulative: bool = False) -> None:
  histograms, values = exposure.histogram(image, channel_axis=2)
  for histogram, color in zip(histograms, ("red", "green", "blue")):
    if cumulative:
      histogram = np.cumsum(histogram)
    plt.plot(values, histogram, color=color)
  plt.xlabel("Intensity")
  plt.ylabel("Number of pixels")
  plt.show()


im = io.imread("la-havana.jpg")
print(im.shape, im.dtype)
show_histogram(im)
show_histogram(im, cumulative=True)

## 2. Correcting the Contrast

The `exposure` module provides several histogram-based contrast corrections. Apply three of them to `im`; display each result next to the original with `side_by_side`, then its histogram and its cumulative histogram.

- `exposure.equalize_hist` equalizes the histogram of the whole image, a non-linear correction: each intensity is replaced by the share of pixels of lower or equal intensity, which brings the cumulative histogram close to a straight line. It's the equalization the Pillow and NumPy lab writes by hand, on a grayscale image: compare the cumulative histogram before and after. On a color image, the function shows a warning (`UserWarning: This might be a color image…`): it computes a single histogram, on the values of the three channels put together, and applies the same transformation to all three, which can shift the colors.
- `exposure.equalize_adapthist` equalizes the histogram region by region (CLAHE, *Contrast Limited Adaptive Histogram Equalization*): the contrast adapts to each part of the image.
- `exposure.rescale_intensity` linearly stretches a range of intensities over the whole range. Choose the range with `np.percentile`: from the intensity below which 2% of the pixels lie to the one below which 98% lie.

Which correction renders the sky and its clouds best? The road? Which one keeps the car's colors?

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_hist
- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_adapthist
- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.rescale_intensity
- https://numpy.org/doc/stable/reference/generated/numpy.percentile.html

In [ ]:
# Your code here

### Additional hints

- `exposure.equalize_hist(im)` and `exposure.equalize_adapthist(im)` return floats between 0 and 1, which `imshow` and `show_histogram` accept as they are.
- `np.percentile(im, (2, 98))` returns the range's two bounds, 41 and 233 for this photo.
- `exposure.rescale_intensity(im, in_range=(low, high))` maps `low` to the smallest value of the image's type (0 for `uint8`) and `high` to the largest (255); the intensities outside the range are clipped to these bounds.
- `side_by_side((im, im_equalized), subtitles=("Original", "Equalized"))` displays two images side by side.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
im_equalized = exposure.equalize_hist(im)
side_by_side((im, im_equalized), subtitles=("Original", "Equalized"))
show_histogram(im_equalized)
show_histogram(im_equalized, cumulative=True)

im_adaptive = ...
...

low, high = np.percentile(im, ...)
im_stretched = exposure.rescale_intensity(im, in_range=...)
...
```

### Solution

In [ ]:
im_equalized = exposure.equalize_hist(im)
side_by_side((im, im_equalized), subtitles=("Original", "Equalized"))
show_histogram(im_equalized)
show_histogram(im_equalized, cumulative=True)

In [ ]:
im_adaptive = exposure.equalize_adapthist(im)
side_by_side((im, im_adaptive), subtitles=("Original", "Adaptive equalization"))
show_histogram(im_adaptive)
show_histogram(im_adaptive, cumulative=True)

In [ ]:
# Stretch the intensities between the 2nd and 98th percentiles over the whole range.
low, high = np.percentile(im, (2, 98))
im_stretched = exposure.rescale_intensity(im, in_range=(low, high))
side_by_side((im, im_stretched), subtitles=("Original", "Stretched intensities"))
show_histogram(im_stretched)
show_histogram(im_stretched, cumulative=True)

# Global equalization darkens the sky and makes the clouds stand out, but
# darkens the road and shifts the colors: the car turns bluer, the sidewalk
# yellower. Adaptive equalization brings out the clouds' relief and the road's
# grain, without changing the colors. The linear stretch, gentler, keeps
# natural colors.

## 3. Detecting Edges, Before and After a Blur

The Sobel filter measures, at each pixel, how much the intensity varies around it: high on an edge, zero in a uniform area. But a photo's grain, the asphalt's for instance, is made of small variations too, which Sobel takes for edges. A Gaussian blur erases them before the detection.

- Convert `im` to grayscale into `im_gray`, and apply `filters.sobel` to it, into `raw_edges`.
- Blur `im_gray` with `filters.gaussian`, with a standard deviation `sigma=2`, into `im_blurred`, then apply `filters.sobel` to the blurred image, into `blurred_edges`.
- Display both results side by side with `side_by_side`, then a detail of both, the car: rows 800 onwards, columns 450 to 1200.
- Measure the edges' noise: the mean of each result over the road, the image's last 100 rows, where nearly everything Sobel finds is the asphalt's grain.

Which edges does the blur erase? Which ones does it keep, and what does it do to them? Try again with `sigma=1`, then with `sigma=4`: what do you gain, what do you lose?

In [ ]:
# Your code here

### Additional hints

- `color.rgb2gray(im)` returns floats between 0 and 1, of shape `(1247, 1920)`; `filters.gaussian` and `filters.sobel` return floats too.
- The larger `sigma`, the stronger the blur.
- `raw_edges[800:, 450:1200]` is the car's detail, `raw_edges[-100:]` the last 100 rows, and `.mean()` gives an array's mean. You should find about 0.076 without the blur, and 0.008 with it: the blur divides the noise by ten.
- `side_by_side` displays a single-channel image in grayscale, its highest value in white.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
im_gray = color.rgb2gray(im)
raw_edges = filters.sobel(im_gray)
im_blurred = filters.gaussian(im_gray, sigma=...)
blurred_edges = ...

subtitles = ("Sobel", "Gaussian blur, then Sobel")
side_by_side((raw_edges, blurred_edges), subtitles=subtitles)
side_by_side((raw_edges[800:, 450:1200], ...), subtitles=subtitles)
print(raw_edges[...].mean(), blurred_edges[...].mean())
```

### Solution

In [ ]:
im_gray = color.rgb2gray(im)
raw_edges = filters.sobel(im_gray)
# The blur erases the grain before the edge detection.
im_blurred = filters.gaussian(im_gray, sigma=2)
blurred_edges = filters.sobel(im_blurred)

subtitles = ("Sobel", "Gaussian blur, then Sobel")
side_by_side((raw_edges, blurred_edges), subtitles=subtitles)
# A detail: the car.
side_by_side(
  (raw_edges[800:, 450:1200], blurred_edges[800:, 450:1200]),
  subtitles=subtitles,
)
# The edges' noise on the road, the last 100 rows.
print(raw_edges[-100:].mean(), blurred_edges[-100:].mean())

# Without the blur, the road is covered with small edges, those of the
# asphalt's grain. After the blur, almost none are left (a mean ten times
# lower), while the sharp edges, those of the bodywork, the windows and the
# street lamps, remain, thicker and less precise. With sigma=1, part of the
# grain remains; with sigma=4, it disappears, but so do the fine details: the
# license plate and the wheel rims fade away.

## 4. Thresholding, Counting and Measuring Objects

Counting objects and measuring them is a common image analysis: cells, grains, coins. The simplest pipeline has three steps: a thresholding separates the objects from the background, the labeling numbers the connected regions of the resulting mask, then each region is measured. The Malecón photo doesn't lend itself to it: this exercise uses the `data.coins()` image, shipped with scikit-image, which shows 24 Greek coins on a gray background.

- Load the image into `coins`.
- Compute its Otsu threshold, `threshold`, with `filters.threshold_otsu`, then the mask `mask`, true for the pixels brighter than the threshold. Display the image and the mask side by side.
- Label the mask's regions with `measure.label`, into `labels`, and display the number of regions found.
- Measure the regions with `measure.regionprops`, and keep in `large_regions` those with an area (`area`) of at least 100 pixels: the others are specks of the background. Display their number.
- Display the image with, on each large region, a cross at its center (`centroid`) and its area.

Do you find as many large regions as coins? Is each region a coin? Look at the largest one.

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.coins
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.text.html

In [ ]:
# Your code here

### Additional hints

- `data.coins()` returns a grayscale image, a `uint8` array of shape `(303, 384)`.
- `filters.threshold_otsu(coins)` returns the threshold, 107 here, and `coins > threshold` the mask, an array of booleans.
- `measure.label(mask)` gives each pixel of the mask the number of its region, from 1 to the number of regions, and 0 to the background: `labels.max()` is the number of regions, 96.
- `measure.regionprops(labels)` returns a list of objects, one per region, with their measures as attributes: `area`, the area in pixels, and `centroid`, the center, a `(row, column)` pair. You should keep 24 regions.
- `plt.scatter(column, row, marker="+")` and `plt.text(column, row, text)` place a cross and a text: the x-coordinate, the column, before the y-coordinate, the row.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
coins = data.coins()
threshold = filters.threshold_otsu(...)
mask = ...
side_by_side((coins, mask), subtitles=(f"Otsu threshold: {threshold}", "Mask"))

labels = measure.label(...)
print(labels.max(), "regions")

large_regions = [region for region in measure.regionprops(labels) if ...]
print(len(large_regions), "regions of at least 100 pixels")

plt.figure(dpi=150)
plt.imshow(coins, cmap="gray")
for region in large_regions:
  row, column = region.centroid
  plt.scatter(..., ..., marker="+", color="red")
  plt.text(column, row, str(int(region.area)), color="yellow", fontsize=6)
plt.axis("off")
plt.show()
```

### Solution

In [ ]:
coins = data.coins()
threshold = filters.threshold_otsu(coins)
# True for the pixels brighter than the threshold: the coins, on a dark background.
mask = coins > threshold
side_by_side((coins, mask), subtitles=(f"Otsu threshold: {threshold}", "Mask"))

# One number per connected region, 0 for the background.
labels = measure.label(mask)
print(labels.max(), "regions")

# The small regions are specks of the background brighter than the threshold.
large_regions = [region for region in measure.regionprops(labels) if region.area >= 100]
print(len(large_regions), "regions of at least 100 pixels")

plt.figure(dpi=150)
plt.imshow(coins, cmap="gray")
for region in large_regions:
  # The center is a (row, column) pair: the column is the x-coordinate.
  row, column = region.centroid
  plt.scatter(column, row, marker="+", color="red")
  plt.text(column, row, str(int(region.area)), color="yellow", fontsize=6)
plt.axis("off")
plt.show()

# 24 large regions for 24 coins: the count is right, but by chance. The top
# left coin touches a bright band along the top edge, and forms with it the
# largest region (8792 pixels), whose center falls on no coin: 23 regions are
# single coins, the 24th mixes a coin and the band. The areas range from single
# to triple: the coins aren't all the same size, and the threshold leaves holes
# in the darkest ones. Always check the result on the image, not only the
# number of objects.

## 5. Detecting Keypoints

A keypoint detector finds remarkable points in an image, such as corners, and describes the neighborhood of each one with a descriptor, a signature that can be compared from one image to another. ORB (*Oriented FAST and Rotated BRIEF*) is one of them, designed to find the same points in a rotated image.

- Rotate `im` by 45° with `transform.rotate`, as in the course, into a new image `im_rotated`, and display both images side by side.
- Create a `feature.ORB` detector, and call its `detect_and_extract` method on `im` in grayscale: the detector then stores the points it found in its `keypoints` attribute, and their descriptors in `descriptors`. Keep them in `keypoints_1` and `descriptors_1`.
- Do the same with `im_rotated`, into `keypoints_2` and `descriptors_2`.
- Display each image with its keypoints on top.

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.ORB
- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.ORB.detect_and_extract
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html

In [ ]:
# Your code here

### Additional hints

- `detect_and_extract` expects a single-channel image: `color.rgb2gray(im)`.
- The same detector works for both images: its `keypoints` and `descriptors` attributes are replaced on each call, read them before the next call.
- `keypoints` is an array of shape `(500, 2)`: ORB keeps 500 points by default, each a `(row, column)` pair. `descriptors` has one row of 256 booleans per point.
- `plt.scatter(x, y)` expects the x-coordinate, the column, before the y-coordinate, the row: `points[:, 1]`, then `points[:, 0]`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
im_rotated = transform.rotate(im, ...)
side_by_side((im, im_rotated), subtitles=("Original", "Rotated by 45°"))

detector = feature.ORB()
detector.detect_and_extract(color.rgb2gray(im))
keypoints_1 = detector.keypoints
descriptors_1 = ...

detector.detect_and_extract(...)
keypoints_2 = ...
descriptors_2 = ...

for image, points in ((im, keypoints_1), (im_rotated, keypoints_2)):
  plt.imshow(image)
  plt.scatter(..., ..., s=5)
  plt.axis("off")
  plt.show()
```

### Solution

In [ ]:
im_rotated = transform.rotate(im, 45)
side_by_side((im, im_rotated), subtitles=("Original", "Rotated by 45°"))

In [ ]:
detector = feature.ORB()

# The image's keypoints and descriptors, in grayscale.
detector.detect_and_extract(color.rgb2gray(im))
keypoints_1 = detector.keypoints
descriptors_1 = detector.descriptors

# The detector replaces its attributes on each call.
detector.detect_and_extract(color.rgb2gray(im_rotated))
keypoints_2 = detector.keypoints
descriptors_2 = detector.descriptors

for image, points in ((im, keypoints_1), (im_rotated, keypoints_2)):
  plt.imshow(image)
  # The points are (row, column) pairs: the column is the x-coordinate.
  plt.scatter(points[:, 1], points[:, 0], s=5)
  plt.axis("off")
  plt.show()

## 6. Matching Keypoints

`feature.match_descriptors` compares the descriptors of two images, and pairs each point of the first one with the point of the second whose descriptor resembles it most. You'll need it twice, here and to stitch two images in the next exercise: write the function `match_keypoints(image_1, image_2, max_ratio=1.0)`, which detects with ORB the keypoints of both images and their descriptors, as in the previous exercise, matches the descriptors with `feature.match_descriptors`, and returns the keypoints of the first image, those of the second, and the matches.

Apply it to `im` and `im_rotated`, into `keypoints_1`, `keypoints_2` and `matches`, then display the matches between the two images with `feature.plot_matched_features`. Do the points of the buildings and street lamps land in the right place in the rotated image?

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors
- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.plot_matched_features

In [ ]:
def match_keypoints(
  image_1: np.ndarray, image_2: np.ndarray, max_ratio: float = 1.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  # Your code here
  ...


# Your code here

### Additional hints

- The previous exercise's detection code becomes the start of the function, applied to `image_1` then to `image_2`.
- `feature.match_descriptors(descriptors_1, descriptors_2, max_ratio=max_ratio)` returns a two-column array, one row per pair: the point's index in `keypoints_1`, then its match's index in `keypoints_2`. Binary descriptors, like ORB's, are compared by default with the Hamming distance, the number of bits that differ. `max_ratio=1.0`, its default value, keeps every pair; the next exercise uses it to drop the ambiguous pairs.
- You should get 320 matches between `im` and `im_rotated`.
- `feature.plot_matched_features(image_1, image_2, keypoints0=..., keypoints1=..., matches=..., ax=ax)` draws the two images side by side and links the points of each pair; `ax` comes from `fig, ax = plt.subplots(dpi=200)`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def match_keypoints(
  image_1: np.ndarray, image_2: np.ndarray, max_ratio: float = 1.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  detector = feature.ORB()
  detector.detect_and_extract(color.rgb2gray(image_1))
  keypoints_1 = detector.keypoints
  descriptors_1 = detector.descriptors

  detector.detect_and_extract(...)
  keypoints_2 = ...
  descriptors_2 = ...

  matches = feature.match_descriptors(..., ..., max_ratio=...)
  return keypoints_1, keypoints_2, matches


keypoints_1, keypoints_2, matches = match_keypoints(..., ...)
print(len(matches), "matches")

fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im, im_rotated, keypoints0=..., keypoints1=..., matches=matches, ax=ax
)
ax.axis("off")
plt.show()
```

### Solution

In [ ]:
def match_keypoints(
  image_1: np.ndarray, image_2: np.ndarray, max_ratio: float = 1.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  detector = feature.ORB()
  detector.detect_and_extract(color.rgb2gray(image_1))
  keypoints_1 = detector.keypoints
  descriptors_1 = detector.descriptors

  # The detector replaces its attributes at each call.
  detector.detect_and_extract(color.rgb2gray(image_2))
  keypoints_2 = detector.keypoints
  descriptors_2 = detector.descriptors

  # One row per pair: the point's index in keypoints_1, then in keypoints_2.
  matches = feature.match_descriptors(descriptors_1, descriptors_2, max_ratio=max_ratio)
  return keypoints_1, keypoints_2, matches


keypoints_1, keypoints_2, matches = match_keypoints(im, im_rotated)
print(len(matches), "matches")

In [ ]:
fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im, im_rotated, keypoints0=keypoints_1, keypoints1=keypoints_2, matches=matches, ax=ax
)
ax.axis("off")
plt.show()

## 7. Stitching Two Overlapping Images

Two photos taken next to each other share a part: the keypoints they have in common tell how far one is shifted from the other, hence where to stitch them. That's how a panorama works. You'll stitch two overlapping pieces of `im` back together, which lets you check the result: stitched, they must give `im` back.

### Splitting the Image

Write the function `split_in_two(image, overlap)`, which cuts `image` into two pieces sharing `overlap` columns, in the middle of the image: the left piece goes from the image's left edge to the end of the shared area, the right one from the start of the shared area to the right edge. NumPy slices are enough.

Apply it to `im`, with an overlap of 960 columns, half its width, to get two images `im_1` and `im_2`. Display their shapes, then both images side by side. Both pieces should have the shape `(1247, 1440, 3)`.

In [ ]:
def split_in_two(image: np.ndarray, overlap: int) -> tuple[np.ndarray, np.ndarray]:
  # Your code here
  ...


# Your code here

#### Additional hints

- `image.shape[1]` is the image's width. Centered, the shared area starts at column `start = (width - overlap) // 2`, and stops before column `start + overlap`: from 480 to 1440 for `im`.
- `image[:, :end]` keeps the columns with an index lower than `end`, and `image[:, start:]` those from `start` on, with every row and every channel.
- `im_1` is `im` without its last 480 columns, `im_2` without its first 480.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def split_in_two(image: np.ndarray, overlap: int) -> tuple[np.ndarray, np.ndarray]:
  width = image.shape[1]
  start = ...
  end = start + overlap
  return image[:, :end], image[:, ...]


im_1, im_2 = split_in_two(im, ...)
print(im_1.shape, im_2.shape)
side_by_side((im_1, im_2))
```

#### Solution

In [ ]:
def split_in_two(image: np.ndarray, overlap: int) -> tuple[np.ndarray, np.ndarray]:
  width = image.shape[1]
  # The shared area, in the middle: from column start to column end, excluded.
  start = (width - overlap) // 2
  end = start + overlap
  return image[:, :end], image[:, start:]


im_1, im_2 = split_in_two(im, 960)
print(im_1.shape, im_2.shape)
side_by_side((im_1, im_2))

### Matching the Two Images' Keypoints

Match the keypoints of `im_1` and `im_2` with your `match_keypoints` function, into `keypoints_1`, `keypoints_2` and `matches`, keeping only the unambiguous pairs thanks to its `max_ratio` parameter. Display the matches.

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors

In [ ]:
# Your code here

#### Additional hints

- `max_ratio=0.8` drops a pair when the second-best candidate resembles the point almost as much as the best one, the ratio of their distances exceeding 0.8: these ambiguous pairs are often wrong.
- You should get 256 matches.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
keypoints_1, keypoints_2, matches = match_keypoints(im_1, im_2, max_ratio=...)
print(len(matches), "matches")

fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im_1, im_2, keypoints0=..., keypoints1=..., matches=matches, ax=ax
)
ax.axis("off")
plt.show()
```

#### Solution

In [ ]:
# The ambiguous pairs dropped: those where the ratio of the distances to the best
# candidate and to the second best exceeds 0.8.
keypoints_1, keypoints_2, matches = match_keypoints(im_1, im_2, max_ratio=0.8)
print(len(matches), "matches")

In [ ]:
fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im_1, im_2, keypoints0=keypoints_1, keypoints1=keypoints_2, matches=matches, ax=ax
)
ax.axis("off")
plt.show()

### Size of the Shared Area

- Compute the average gap, in columns, between the two points of each pair, into `shift`: it's the number of pixels by which `im_2` is shifted from `im_1`.
- Deduce the width of the area the two images share, `zone_size`, in pixels: the width of `im_1` minus that shift. Round it to the nearest integer: a number of pixels is an integer, and truncating with `int` would leave one column too many to stitch as soon as the shift exceeds an integer by a few thousandths.

In [ ]:
# Your code here

#### Additional hints

- `keypoints_1[matches[:, 0]]` and `keypoints_2[matches[:, 1]]` put the points of each pair face to face. The absolute value of their difference, averaged over axis 0 (`.mean(axis=0)`), gives the average gap in rows, then in columns: only the columns' one matters.
- `round(x)` rounds to the nearest integer.
- You should find a shift of about 480 pixels, what `split_in_two` removed from each side, and a shared area of 960 pixels: the overlap given to `split_in_two`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
_, shift = np.abs(keypoints_2[...] - keypoints_1[...]).mean(axis=0)
zone_size = round(...)
print(shift, zone_size)
```

#### Solution

In [ ]:
# The average gap between the points of each pair, in rows and in columns.
_, shift = np.abs(keypoints_2[matches[:, 1]] - keypoints_1[matches[:, 0]]).mean(axis=0)
# Rounded, not truncated: int(1440 - 480.002) would give 959.
zone_size = round(im_1.shape[1] - shift)
print(shift, zone_size)

### Stitching the Two Images

Remove from `im_2` its left part, as wide as the shared area, then put the rest to the right of `im_1` with `np.concatenate`, into `im_merged`. Display the result, and check that it gives `im` back.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html
- https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html

In [ ]:
# Your code here

#### Additional hints

- `im_2[:, zone_size:, :]` removes the first `zone_size` columns of `im_2`.
- `np.concatenate((a, b), axis=1)` puts `b` to the right of `a`: axis 1 is the columns' one.
- `im_merged` must have the shape of `im`, `(1247, 1920, 3)`, and `np.array_equal(im_merged, im)` must return `True`: the two stitched images give back exactly the original photo.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
im_merged = np.concatenate((im_1, ...), axis=...)
print(im_merged.shape, np.array_equal(im_merged, im))

plt.figure(dpi=100)
plt.imshow(im_merged)
plt.axis("off")
plt.show()
```

#### Solution

In [ ]:
im_merged = np.concatenate((im_1, im_2[:, zone_size:, :]), axis=1)
print(im_merged.shape, np.array_equal(im_merged, im))

plt.figure(dpi=100)
plt.imshow(im_merged)
plt.axis("off")
plt.show()

### Going Further: Stitching With a Fade (Demonstration)

From one photo to the next, the light changes, and a sharp cut leaves a visible edge. The `stitch_with_fade` function below stitches the two images with a fade instead: in the middle of the shared area, over `2 * fade_width` columns, the weight of `im_2` goes from 0 to 1 along a linear ramp, broadcast over the rows and channels, like the `alpha` ramp of the "Image blending" exercise of the Pillow and NumPy lab. The cell tries it on a darkened `im_2`, like a second photo taken under a different light.

Run it, and look at the edge between the two images: a sharp step with the cut, a gradual transition with the fade.

In [ ]:
def stitch_with_fade(
  image_1: np.ndarray, image_2: np.ndarray, zone_size: int, fade_width: int = 30
) -> np.ndarray:
  # The shared area: the end of image_1, the start of image_2.
  end_1 = image_1[:, -zone_size:, :]
  start_2 = image_2[:, :zone_size, :]
  # The weight of image_2 in the shared area: 0 on the left, 1 on the right,
  # and a linear ramp over 2 * fade_width columns, in the middle.
  weights = np.zeros(zone_size)
  middle = zone_size // 2
  weights[middle:] = 1
  weights[middle - fade_width : middle + fade_width] = np.linspace(0, 1, 2 * fade_width)
  # One weight per column: the shape (1, zone_size, 1) is broadcast over the
  # rows and the channels.
  weights = weights[np.newaxis, :, np.newaxis]
  zone = end_1 * (1 - weights) + start_2 * weights
  return (
    np.concatenate(
      (image_1[:, :-zone_size, :], zone, image_2[:, zone_size:, :]), axis=1
    )
    .round()
    .astype(np.uint8)
  )


# im_2 darkened, like a second photo taken under a different light.
im_2_dark = (im_2 * 0.7).astype(np.uint8)
cut = np.concatenate((im_1, im_2_dark[:, zone_size:, :]), axis=1)
faded = stitch_with_fade(im_1, im_2_dark, zone_size, fade_width=200)
side_by_side((cut, faded), subtitles=("Sharp cut", "Fade"), dpi=200)

## 8. Image Comparison Metrics

A comparison metric measures how much two images resemble each other. Prepare three modified versions of the photo:

- Convert `im` to floats between 0 and 1 into `im_float`, with `util.img_as_float` as in the course, or yourself.
- Create a noisy image `im_noisy`: add to `im_float` a Gaussian noise of the same shape, drawn by `np.random.normal`, then bring the values back between 0 and 1 with `np.clip`.
- Create a brightened image `im_brightened`: `im_float` plus a constant, brought back between 0 and 1 the same way.
- Create the mirror image `im_mirror`, the left-right reflection of `im_float` (a symmetry about a vertical axis).

Then compare `im_float` with itself and with each of the three others using two metrics, and display the four images side by side, with the metrics' values as subtitles:

- `metrics.structural_similarity` (SSIM), designed to follow our perception of resemblance: between -1 and 1, 1 for two identical images;
- `metrics.mean_squared_error` (MSE), the mean squared error: from 0, for two identical images, to infinity.

Which image does the MSE find closest to the original? And the SSIM? Which of the two agrees with your eye?

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.mean_squared_error
- https://numpy.org/doc/stable/reference/random/generated/numpy.random.normal.html
- https://numpy.org/doc/stable/reference/generated/numpy.clip.html

In [ ]:
# Your code here

### Additional hints

- `np.random.normal(0, 0.2, im_float.shape)` draws a noise of mean 0 and standard deviation 0.2, of the image's shape; `np.clip(array, 0, 1)` brings what is below 0 to 0, what is above 1 to 1.
- `im_float[:, ::-1]` reverses the order of the columns.
- `metrics.structural_similarity(a, b, channel_axis=2, data_range=1)`: `channel_axis` is the channels' axis, `data_range` the gap between the smallest and largest possible values, 1 for floats between 0 and 1. `metrics.mean_squared_error(a, b)` needs no parameter.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
im_float = util.img_as_float(im)
im_noisy = np.clip(im_float + np.random.normal(0, 0.2, im_float.shape), 0, 1)
im_brightened = ...
im_mirror = ...

images = (im_float, im_noisy, im_brightened, im_mirror)
subtitles = []
for image in images:
  mse = metrics.mean_squared_error(im_float, image)
  ssim = ...
  subtitles.append(f"MSE: {mse:.3f}, SSIM: {ssim:.2f}")
side_by_side(images, subtitles=subtitles, dpi=200)
```

### Solution

In [ ]:
im_float = util.img_as_float(im)
noise = np.random.normal(0, 0.2, im_float.shape)
im_noisy = np.clip(im_float + noise, 0, 1)
im_brightened = np.clip(im_float + 0.2, 0, 1)
# The left-right reflection: the columns in reverse order.
im_mirror = im_float[:, ::-1]

images = (im_float, im_noisy, im_brightened, im_mirror)
subtitles = []
for image in images:
  mse = metrics.mean_squared_error(im_float, image)
  ssim = metrics.structural_similarity(im_float, image, channel_axis=2, data_range=1)
  subtitles.append(f"MSE: {mse:.3f}, SSIM: {ssim:.2f}")
side_by_side(images, subtitles=subtitles, dpi=200)

# The three images have a similar MSE, around 0.04, and the MSE even finds the
# noisy image (about 0.035) slightly closer to the original than the brightened
# one (about 0.038). The SSIM ranks them as the eye does: the brightened image,
# which keeps the photo's structure, is the closest (about 0.93), the noisy one
# the farthest (about 0.13), the mirror in between (0.50).